# LeakShield 基础使用示例

本笔记本演示 LeakShield 的基本用法。

In [ ]:
# 安装 LeakShield（如果尚未安装）
# !pip install leakshield

In [ ]:
import numpy as np
import pandas as pd
import leakshield as ls

## 1. 创建示例数据集

In [ ]:
# 训练集
np.random.seed(42)
train_df = pd.DataFrame({
    **{f'feature_{i}': np.random.randn(1000) for i in range(10)},
    'label': np.random.randint(0, 2, 1000)
})

# 测试集（包含 10% 重叠样本）
overlap_df = train_df.iloc[:100].copy()
np.random.seed(123)
new_df = pd.DataFrame({
    **{f'feature_{i}': np.random.randn(100) for i in range(10)},
    'label': np.random.randint(0, 2, 100)
})
test_df = pd.concat([overlap_df, new_df], ignore_index=True)

print(f"训练集形状: {train_df.shape}")
print(f"测试集形状: {test_df.shape}")

## 2. 执行泄露检测（三行代码）

In [ ]:
# 三行代码完成检测
result = ls.check(train_df, test_df)
result.report()

## 3. 检查检测结果

In [ ]:
# 检查是否有泄露
if result:
    print(f"检测到 {len(result)} 项泄露风险")
    print(f"综合风险等级: {result.overall_level}")
    print(f"综合风险分数: {result.overall_score:.2f}")
else:
    print("未检测到数据泄露")

## 4. 导出结果

In [ ]:
# 导出为 JSON
result.to_json('leakage_report.json')

# 或转换为字典
result_dict = result.to_dict()
print(result_dict)

## 5. 自定义配置

In [ ]:
# 使用自定义配置
config = ls.DetectionConfig(
    task_type='classification',
    hash_similarity_threshold=0.95,
    enable_mdf=False  # 仅使用 Hash 引擎
)

result = ls.check(train_df, test_df, config)
result.report()